# How to use it?

This encoding will be integrated into the training and inference process, the encoder.pkl file will be part of the trained model's output. This is just an example for development and debugging purposes.

## Encoder

In [2]:
# Import iris dataset to make the encoding
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

rng = np.random.default_rng(42)
perm = rng.permutation(len(y))
cut = int(0.8 * len(y))  # 80/20

idx_train, idx_test = perm[:cut], perm[cut:]

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

print(X_train.shape, X_test.shape)

(120, 4) (30, 4)


In [2]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from nsbc.utils import encoders

encoder = encoders.MLBinaryEncoderVectorized(encoder_type='gray', verbose=True)
X_train_encoded = encoder.fit_transform(X_train, num_decimals=2)

Fitting GrayCode encoder on 120 samples with 4 features
Normalization learned:
  - Decimal factor: 100
  - Offset value: 0
  - Feature ranges: min=10, max=790
  - Bit widths per feature: [10  9 10  8]
  - Total bit width: 37
Transforming 120 samples


Encoding with GrayCode: 100%|██████████| 1/1 [00:00<?, ?it/s]

Output shape: (120, 37)


In [3]:
# Save parameters for later use
encoder.save_params("encoder_exp.pkl")

Encoding parameters saved to encoder_exp.pkl


Example of loading encoder for test data

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from nsbc.utils import encoders
encoder = encoders.MLBinaryEncoderVectorized(encoder_type='gray', verbose=True)
# load parameters to use
encoder.load_params("encoder_exp.pkl")

Encoding parameters loaded from encoder_exp.pkl


In [3]:
# Transform test data using same parameters
X_test_encoded = encoder.transform(X_test)

Transforming 30 samples


Encoding with GrayCode: 100%|██████████| 1/1 [00:00<?, ?it/s]

Output shape: (30, 37)


### Optimazing hamming distance

In [ ]:
import numpy as np
import time

def hamming_sum(xi: np.ndarray, xw: np.ndarray) -> int:
    """Original version with sum"""
    return np.sum(np.bitwise_xor(xi, xw))

def hamming_count_nonzero(xi: np.ndarray, xw: np.ndarray) -> int:
    """Your version with count_nonzero"""
    return np.count_nonzero(np.bitwise_xor(xi, xw))

def hamming_not_equal(xi: np.ndarray, xw: np.ndarray) -> int:
    """Even faster: using != operator"""
    return np.count_nonzero(xi != xw)

def hamming_sum_not_equal(xi: np.ndarray, xw: np.ndarray) -> int:
    """Alternative with sum"""
    return np.sum(xi != xw)


# Benchmark
sizes = [100, 1000, 10000, 100000, 1000000]

for size in sizes:
    xi = np.random.randint(0, 2, size, dtype=np.uint8)
    xw = np.random.randint(0, 2, size, dtype=np.uint8)
    
    n_iterations = 10000
    
    # Test np.sum + bitwise_xor
    start = time.time()
    for _ in range(n_iterations):
        _ = hamming_sum(xi, xw)
    t1 = time.time() - start
    
    # Test np.count_nonzero + bitwise_xor
    start = time.time()
    for _ in range(n_iterations):
        _ = hamming_count_nonzero(xi, xw)
    t2 = time.time() - start
    
    # Test np.count_nonzero + !=
    start = time.time()
    for _ in range(n_iterations):
        _ = hamming_not_equal(xi, xw)
    t3 = time.time() - start
    
    # Test np.sum + !=
    start = time.time()
    for _ in range(n_iterations):
        _ = hamming_sum_not_equal(xi, xw)
    t4 = time.time() - start
    
    print(f"\n=== Size: {size} ===")
    print(f"sum + bitwise_xor:         {t1:.4f}s (baseline)")
    print(f"count_nonzero + bitwise_xor: {t2:.4f}s ({t1/t2:.2f}x faster)")
    print(f"count_nonzero + !=:        {t3:.4f}s ({t1/t3:.2f}x faster)")
    print(f"sum + !=:                  {t4:.4f}s ({t1/t4:.2f}x faster)")


=== Size: 100 ===
sum + bitwise_xor:         0.0520s (baseline)
count_nonzero + bitwise_xor: 0.0115s (4.52x faster)
count_nonzero + !=:        0.0110s (4.73x faster)
sum + !=:                  0.0290s (1.79x faster)

=== Size: 1000 ===
sum + bitwise_xor:         0.0410s (baseline)
count_nonzero + bitwise_xor: 0.0145s (2.83x faster)
count_nonzero + !=:        0.0110s (3.73x faster)
sum + !=:                  0.0340s (1.21x faster)

=== Size: 10000 ===
sum + bitwise_xor:         0.1309s (baseline)
count_nonzero + bitwise_xor: 0.0575s (2.28x faster)
count_nonzero + !=:        0.0160s (8.18x faster)
sum + !=:                  0.0895s (1.46x faster)

=== Size: 100000 ===
sum + bitwise_xor:         0.8319s (baseline)
count_nonzero + bitwise_xor: 0.4806s (1.73x faster)
count_nonzero + !=:        0.0645s (12.89x faster)
sum + !=:                  0.5507s (1.51x faster)

=== Size: 1000000 ===
sum + bitwise_xor:         7.9869s (baseline)
count_nonzero + bitwise_xor: 4.7558s (1.68x faster)
coun

## Test Training part

In [1]:
# Import iris dataset to make the encoding
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

rng = np.random.default_rng(42)
perm = rng.permutation(len(y))
cut = int(0.8 * len(y))  # 80/20

idx_train, idx_test = perm[:cut], perm[cut:]

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

print(X_train.shape, X_test.shape)

(120, 4) (30, 4)


In [2]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from nsbc.engine import NSBCEngine

In [3]:
# Initialize and train the model
model_nsbc = NSBCEngine(n_value=1, decimals=2)
model_nsbc.fit(X_train, y_train)

100%|██████████| 4/4 [00:00<?, ?it/s]


In [4]:
model_nsbc.save()

In [5]:
import pickle
file_path = 'model.pkl'  # Replace with the actual path to your PKL file
with open(file_path, 'rb') as file:
    data = pickle.load(file)

### Testing trained model